In [0]:
# =============================================================
# NOTEBOOK:  09_bronze_api_pagination
# PURPOSE:   Learn paginated API ingestion
# SOURCE:    PokeAPI (free, no auth needed)
# TARGET:    Bronze Delta Table
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
import time
import uuid
import requests
from pyspark.sql import functions as F

# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("=" * 60)
print("RetailMax - Bronze: Paginated API Ingestion")
print("=" * 60)

# ── Paths ─────────────────────────────────────────────────────
target_path     = bronze_path + "tables/external_api/pokemon/"
checkpoint_path = bronze_path + "_checkpoints/external_api/pokemon/"

# ── Reusable Retry Helper (from Day 4) ────────────────────────
def call_api_with_retry(
    url,
    timeout=30,
    max_retries=3,
    backoff_seconds=2
):
    RETRYABLE_CODES = [429, 500, 502, 503, 504]
    NON_RETRYABLE_CODES = [400, 401, 403, 404]

    for attempt in range(max_retries):
        wait = backoff_seconds * (2 ** attempt)

        try:
            response = requests.get(url, timeout=timeout)
            
            if response.status_code == 200:
                return response.json()
            
            elif response.status_code in NON_RETRYABLE_CODES:
                raise Exception(f"Non-retryable error: {response.status_code}")
            
            elif response.status_code in RETRYABLE_CODES:
                print(f"Attempt {attempt + 1}/{max_retries}: Retryable error {response.status_code}. Waiting {wait}s...")
                time.sleep(wait)
        
        except requests.exceptions.Timeout:
            print(f"Attempt {attempt + 1}/{max_retries}: Timeout. Waiting {wait}s...")
            time.sleep(wait)
        
        except requests.exceptions.ConnectionError:
            print(f"Attempt {attempt + 1}/{max_retries}: Connection error. Waiting {wait}s...")
            time.sleep(wait)

    raise Exception(f"API call failed after {max_retries} attempts")

print("✅ Setup complete")
print(f"   Target path: {target_path}")

In [0]:
api_url = "https://pokeapi.co/api/v2/pokemon?limit=20&offset=0"

max_pages = 3
page_no = 1
all_results = []

while(api_url is not None and page_no<=max_pages):
  data = call_api_with_retry(api_url)
  page_results = [
      {
          "pokemon_name": pokemon["name"],
          "pokemon_url": pokemon["url"],
          "page_number": page_no
      } for pokemon in data["results"]
  ]
  all_results.extend(page_results)
  print(f"Page {page_no}: fetched {len(data['results'])} records")
  api_url = data["next"]
  page_no += 1

print(f"Total pages: {page_no - 1}")
print(f"Total records: {len(all_results)}")
print("First 3 pokemon names:", [pokemon["pokemon_name"] for pokemon in all_results[:3]])

df = spark.createDataFrame(all_results)

batch_id = uuid.uuid4().hex
df = (df
    .withColumn("_load_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
    .withColumn("_batch_id", F.lit(batch_id))
)

df.write\
    .mode('append')\
    .format('delta')\
    .save(target_path)

df.show(5)
df.printSchema()